# Final_01_pre.ipynb 步驟目錄
---
| Step | 內容 | 使用資料 | 輸出檔案 |
|---:|---|---|---|
| 1 | 載入套件與建立輸出資料夾 | - | - |
| 2 | 讀取或下載臺北市道路路網，並投影至 `EPSG:3826` | OSMnx 臺北市路網 | - |
| 3 | 依 OSM 屬性分類道路型態，只建立 `road_type`，不定義車速 | `G` | `output/taipei_drive.graphml` |
| 4 | 篩選臺北市室內避難所，並配對 500 m 內最近道路 | `data/shelter_marked.csv`, `G` | `output/taipei_shelters_join_500m.geojson` |
| 5 | 建立臺北市 `250 m × 250 m` grid | `data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp` | - |
| 6 | 計算每個 grid 的平均高程與平均坡度 | `data/Taipei_dem.tif`, grid | - |
| 7 | 計算每個 grid 的河川占比與到最近河川距離 | `data/riverpoly/riverpoly.shp`, grid | - |
| 9 | 根據地形、河川、與坡度建立風險指標 | 地形、坡度、河川、歷史淹水結果 | - |
| 10 | 輸出完整風險 grid | grid risk result | `output/Taipei_grid_full_risk.geojson`, `output/Taipei_grid_full_risk.csv` |
| 11 | 建立 pre map 互動式視覺化 | 風險 grid、道路、室內避難所、臺北市邊界 | `output/Taipei_pre_map.html` |
| 12 | 將道路依 `road_type` 編號，並切分到 grid 內產生 `road_grid_id` | `G`, 風險 grid | `output/taipei_road_grid_segments.geojson` |

In [ ]:
import os, json, ast
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from dotenv import load_dotenv

load_dotenv()
os.makedirs("output", exist_ok=True)


### 讀取道路路網與道路型態分類
---
此步驟讀取或下載臺北市道路路網，並將路網投影至 `EPSG:3826`。接著依據 OSM 道路屬性，將每條道路分類為 `tunnel`、`expressway`、`arterial`、`bridge`、`residential` 或 `service`。

本階段只建立道路型態 `road_type`，不定義車速與通行時間，以保持前處理資料乾淨。道路速度與災害後通行時間會在後續 post-disaster analysis 階段處理。

In [ ]:
place_name = "臺北市, 臺灣"
graph_path = "output/taipei_drive.graphml"

# ===== 1) 讀取或下載道路 graph =====
if os.path.exists(graph_path):
    G = ox.load_graphml(graph_path)
else:
    G = ox.graph_from_place(place_name, network_type="drive_service", simplify=True)

G = ox.project_graph(G, to_crs="EPSG:3826")

print("Road graph loaded. Nodes:", len(G.nodes), "Edges:", len(G.edges))


# ===== 2) 道路型態分類函式 =====
def pick_first(val):
    if isinstance(val, list):
        return val[0] if val else None
    if pd.isna(val):
        return None
    s = str(val).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = ast.literal_eval(s)
            if isinstance(arr, list) and arr:
                return arr[0]
        except Exception:
            pass
    return s


def classify_road(data):
    hw = str(pick_first(data.get("highway")) or "").lower()
    bridge = str(pick_first(data.get("bridge")) or "no").lower()
    tunnel = str(pick_first(data.get("tunnel")) or "no").lower()
    current_road_type = str(pick_first(data.get("road_type")) or "").lower()

    if (
        tunnel in {"yes", "true", "1", "building_passage", "underpass"}
        or current_road_type in {"tunnel", "underground", "underground_road", "underpass"}
    ):
        return "tunnel"
    if hw in {"motorway", "motorway_link", "trunk", "trunk_link"}:
        return "expressway"
    if bridge in {"yes", "viaduct"}:
        return "bridge"
    if hw == "service":
        return "service"
    if hw in {"primary", "primary_link", "secondary", "secondary_link"}:
        return "arterial"
    return "residential"


# ===== 3) 寫入 road_type，不處理速度 =====
for _, _, _, data in G.edges(keys=True, data=True):
    data["road_type"] = classify_road(data)

    # 保持 pre stage 乾淨，不保留舊流程產生的速度欄位
    data.pop("speed_kph", None)
    data.pop("travel_time", None)


# ===== 4) 儲存 graph =====
ox.save_graphml(G, graph_path)

print("Saved:", graph_path)

### 避難所篩選與道路配對
---
- 讀取 `data/shelter_marked.csv`
- 清理欄位名稱與文字格式
- 篩選臺北市避難所
- 只保留 `室內 = 是` 的避難所
- 將避難所經緯度轉為空間點資料
- 將避難所投影至道路路網座標系
- 尋找每個避難所 500 公尺內最近道路
- 每個避難所只保留最近的一條道路配對結果
- 輸出 `output/taipei_shelters_join_500m.geojson`

In [ ]:
# shelters 清理 + 台北市篩選 + 室內避難所篩選 + 轉 GDF + 500m 內最近道路配對（合併版）

import pandas as pd
import geopandas as gpd
import osmnx as ox

# 1) 讀取與清理 shelters
shelters = pd.read_csv("data/shelter_marked.csv")
shelters.columns = [c.strip().strip("'").strip('"') for c in shelters.columns]

for c in shelters.columns:
    if shelters[c].dtype == "object":
        shelters[c] = shelters[c].astype(str).str.strip().str.strip("'").str.strip('"')

# 2) 篩選台北市
if "COUNTYNAME" in shelters.columns:
    mask_tp = shelters["COUNTYNAME"].str.contains("臺北市|台北市", na=False, regex=True)
else:
    mask_tp = shelters["縣市及鄉鎮市區"].str.contains("臺北市|台北市", na=False, regex=True)

shelters_tp = shelters[mask_tp].copy()

# 3) 只保留室內避難所
# 若同時有室內與室外空間，只要「室內」為是，就保留。
if "室內" in shelters_tp.columns:
    shelters_tp = shelters_tp[
        shelters_tp["室內"].astype(str).str.strip().eq("是")
    ].copy()
else:
    raise KeyError("找不到欄位：室內")

# 4) 轉 GeoDataFrame（WGS84）
lon_col, lat_col = "經度", "緯度"
shelters_tp_gdf = gpd.GeoDataFrame(
    shelters_tp,
    geometry=gpd.points_from_xy(
        pd.to_numeric(shelters_tp[lon_col], errors="coerce"),
        pd.to_numeric(shelters_tp[lat_col], errors="coerce"),
    ),
    crs="EPSG:4326",
).dropna(subset=["geometry"])

# 5) 轉路網 edges，CRS 對齊
edges_gdf = ox.graph_to_gdfs(G, nodes=False).copy()
shelters_proj = shelters_tp_gdf.to_crs(edges_gdf.crs) if shelters_tp_gdf.crs != edges_gdf.crs else shelters_tp_gdf.copy()

# 6) 每個 shelter 建唯一 ID
shelters_proj = shelters_proj.reset_index(drop=True)
shelters_proj["shelter_id"] = shelters_proj.index

# 7) 500m 最近道路 join
use_cols = [
    c for c in [
        "highway",
        "road_type",
        "length",
        "geometry",
    ]
    if c in edges_gdf.columns
]

shelter_join = gpd.sjoin_nearest(
    shelters_proj,
    edges_gdf[use_cols],
    how="left",
    max_distance=500,
    distance_col="dist_to_edge_m",
)

# 8) 只留 500m 內有道路，且每個 shelter 只取最近一筆
matched_unique = (
    shelter_join[shelter_join["dist_to_edge_m"].notna()]
    .sort_values(["shelter_id", "dist_to_edge_m"])
    .drop_duplicates(subset=["shelter_id"], keep="first")
    .copy()
)

# 9) 輸出
matched_unique.to_file(
    "output/taipei_shelters_join_500m.geojson",
    driver="GeoJSON",
    encoding="utf-8",
)

print("台北市室內 shelter 數:", len(shelters_proj))
print("保留 shelter 數 (500m 內有道路):", len(matched_unique))
print("移除 shelter 數:", len(shelters_proj) - len(matched_unique))

### 臺北市 Grid 建立
---
- 使用 `TOWN_MOI_1140318.shp` 讀取行政區邊界
- 篩選出臺北市範圍
- 將座標系轉為 `EPSG:3826`
- 以臺北市邊界外包範圍建立規則網格
- 每個 grid 大小為 `250 m × 250 m`
- 單一完整網格面積約為 `62,500 m²`
- 將網格與臺北市邊界相交，只保留臺北市範圍內的 grid
- 為每個 grid 建立唯一編號 `grid_id`

In [ ]:
# Cell 1: import
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats
from shapely.geometry import box


In [ ]:
# Cell 2: 建立台北市 250m x 250m grid（EPSG:3826）
town_path = "data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp"
town = gpd.read_file(town_path)
town.columns = [c.strip().strip("'").strip('"') for c in town.columns]

tp = town[town["COUNTYNAME"].astype(str).str.contains("臺北市|台北市", na=False, regex=True)].copy()
tp = tp.to_crs(epsg=3826)
tp_boundary = tp.dissolve()

minx, miny, maxx, maxy = tp_boundary.total_bounds
cell = 250

xs = np.arange(minx, maxx, cell)
ys = np.arange(miny, maxy, cell)
grid = gpd.GeoDataFrame({"geometry": [box(x, y, x+cell, y+cell) for x in xs for y in ys]}, crs="EPSG:3826")

grid_tp = gpd.overlay(grid, tp_boundary[["geometry"]], how="intersection")
grid_tp["grid_id"] = np.arange(1, len(grid_tp) + 1)

print("grid count:", len(grid_tp))


### 地形與坡度分析
---
- 讀取 `data/Taipei_dem.tif`
- 若 DEM 座標系與 grid 不同，將 DEM 重投影至 `EPSG:3826`
- 計算 DEM 的坡度，單位為 degree
- 使用 zonal statistics 計算每個 grid 的平均高程
- 使用 zonal statistics 計算每個 grid 的平均坡度
- 輸出欄位：
  - `mean_elevation`
  - `slope`

In [ ]:
# Cell 3: DEM -> 每格 mean_elevation + slope（degree）
dem_path = "data/Taipei_dem.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1).astype("float32")
    dem_nodata = src.nodata
    dem_transform = src.transform
    dem_crs = src.crs

if dem_crs != grid_tp.crs:
    with rasterio.open(dem_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, grid_tp.crs, src.width, src.height, *src.bounds
        )
        dst = np.empty((height, width), dtype="float32")
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=grid_tp.crs,
            resampling=Resampling.bilinear
        )
        dem = dst
        dem_transform = transform
        dem_nodata = src.nodata

if dem_nodata is not None:
    dem[dem == dem_nodata] = np.nan

cellsize_x = abs(dem_transform.a)
cellsize_y = abs(dem_transform.e)
gy, gx = np.gradient(dem, cellsize_y, cellsize_x)
slope_raster = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2))).astype("float32")

elev_stats = zonal_stats(grid_tp.geometry, dem, affine=dem_transform, stats=["mean"], nodata=np.nan)
slope_stats = zonal_stats(grid_tp.geometry, slope_raster, affine=dem_transform, stats=["mean"], nodata=np.nan)

grid_final = grid_tp.copy()
grid_final["mean_elevation"] = [s["mean"] for s in elev_stats]
grid_final["slope"] = [s["mean"] for s in slope_stats]

print("elevation non-null:", grid_final["mean_elevation"].notna().sum(), "/", len(grid_final))
print("slope non-null:", grid_final["slope"].notna().sum(), "/", len(grid_final))


### 河川疊圖分析
---
- 讀取 `data/riverpoly/riverpoly.shp`
- 將河川資料轉換至 `EPSG:3826`
- 計算每個 grid 的面積 `grid_area_m2`
- 將 grid 與河川圖層進行空間疊圖
- 計算每個 grid 內的河川面積 `river_area_m2`
- 計算河川占比 `river_ratio`
- 判斷 grid 是否包含河川 `has_river`
- 對沒有河川的 grid，計算其到最近河川的距離 `dist_to_river_m`

In [ ]:
# Cell 4: 河川疊圖 -> river_ratio, dist_to_river_m
river_path = "data/riverpoly/riverpoly.shp"
river = gpd.read_file(river_path)
river_3826 = river.to_crs(epsg=3826)

grid_river = grid_tp.to_crs(epsg=3826).copy()
grid_river["grid_area_m2"] = grid_river.geometry.area

inter = gpd.overlay(
    grid_river[["grid_id", "grid_area_m2", "geometry"]],
    river_3826[["geometry"]],
    how="intersection"
)

if len(inter) > 0:
    inter["river_area_m2"] = inter.geometry.area
    river_area = inter.groupby("grid_id", as_index=False)["river_area_m2"].sum()
else:
    river_area = pd.DataFrame({"grid_id": [], "river_area_m2": []})

grid_river = grid_river.merge(river_area, on="grid_id", how="left")
grid_river["river_area_m2"] = grid_river["river_area_m2"].fillna(0.0)
grid_river["river_ratio"] = (grid_river["river_area_m2"] / grid_river["grid_area_m2"]).fillna(0.0)
grid_river["has_river"] = grid_river["river_area_m2"] > 0

grid_river["dist_to_river_m"] = 0.0
need_dist = grid_river["river_ratio"] == 0
if need_dist.any():
    targets = grid_river.loc[need_dist, ["grid_id", "geometry"]].copy()
    nn = gpd.sjoin_nearest(targets, river_3826[["geometry"]], how="left", distance_col="dist_to_river_m")
    dist_map = nn.groupby("grid_id")["dist_to_river_m"].min()
    grid_river.loc[need_dist, "dist_to_river_m"] = grid_river.loc[need_dist, "grid_id"].map(dist_map)

print("has river grids:", int(grid_river["has_river"].sum()))


### 風險指標計算方式

---

本研究針對每個 grid 計算三項風險指標，其中 `terrain_flood_risk` 與 `river_flood_risk` 用於組成 `total_flood_risk`，`slope_risk` 則獨立表示邊坡風險。

為避免後續使用政府公開淹水潛勢資料進行驗證時產生循環論證，本研究不將歷史淹水紀錄納入 `total_flood_risk`。因此，淹水風險主要由地形條件與河川鄰近性建立，再以外部淹水資料驗證其合理性。

| 風險指標 | 使用資料 | 判斷依據 | 分數 |
|---|---|---|---:|
| `terrain_flood_risk` | `mean_elevation` | 高程低於第 25 百分位數 | +2 |
| `terrain_flood_risk` | `mean_elevation` | 高程介於第 25 至第 50 百分位數 | +1 |
| `terrain_flood_risk` | `mean_elevation`, `slope` | 高程低於第 50 百分位數，且坡度小於 15 度 | +1 |
| `river_flood_risk` | `has_river`, `river_ratio` | grid 內有河川，且河川面積占比大於 0.5 | +3 |
| `river_flood_risk` | `has_river`, `river_ratio` | grid 內有河川，且河川面積占比介於 0 至 0.5 | +2 |
| `river_flood_risk` | `dist_to_river_m` | grid 內無河川，但距離最近河川小於 500 m | +1 |
| `slope_risk` | `mean_elevation`, `slope` | 高程高於第 75 百分位數，且坡度介於 10-20 度 | +1 |
| `slope_risk` | `mean_elevation`, `slope` | 高程高於第 75 百分位數，且坡度介於 20-30 度 | +2 |
| `slope_risk` | `mean_elevation`, `slope` | 高程高於第 75 百分位數，且坡度大於 30 度 | +3 |

| 總指標 | 組成方式 |
|---|---|
| `total_flood_risk` | `terrain_flood_risk + river_flood_risk` |
| `slope_risk` | 獨立作為邊坡 / 土石流相關風險指標 |

In [ ]:
# Cell 5: 三項風險指標（地形淹水 / 河川淹水 / 邊坡）

g = grid_final.copy()

# ===== 合併河川資訊 =====
g = g.merge(
    grid_river[
        [
            "grid_id",
            "river_ratio",
            "dist_to_river_m",
            "has_river",
            "river_area_m2",
            "grid_area_m2",
        ]
    ],
    on="grid_id",
    how="left",
)

g["river_ratio"] = g["river_ratio"].fillna(0.0)
g["dist_to_river_m"] = g["dist_to_river_m"].fillna(np.inf)
g["has_river"] = g["has_river"].fillna(False)

# ===== 高程分位數 =====
q25 = g["mean_elevation"].quantile(0.25)
q50 = g["mean_elevation"].quantile(0.50)
q75 = g["mean_elevation"].quantile(0.75)

# ===== 初始化三項指標 =====
g["terrain_flood_risk"] = 0
g["river_flood_risk"] = 0
g["slope_risk"] = 0

# ===== 1) 地形淹水風險 =====
# 低窪地區較容易積水
g.loc[
    g["mean_elevation"] < q25,
    "terrain_flood_risk",
] += 2

g.loc[
    (g["mean_elevation"] >= q25) & (g["mean_elevation"] < q50),
    "terrain_flood_risk",
] += 1

# 低窪且坡度平緩者，更不利排水
lowland = g["mean_elevation"] < q50

g.loc[
    lowland & (g["slope"] < 15),
    "terrain_flood_risk",
] += 1

# ===== 2) 河川淹水風險 =====
# grid 內河川面積比例越高，河川相關淹水風險越高
g.loc[
    g["has_river"] & (g["river_ratio"] > 0.5),
    "river_flood_risk",
] += 3

g.loc[
    g["has_river"] & (g["river_ratio"] > 0) & (g["river_ratio"] <= 0.5),
    "river_flood_risk",
] += 2

# 無河川但距河川 500m 內，仍給予較低風險分數
g.loc[
    (~g["has_river"]) & (g["dist_to_river_m"] < 500),
    "river_flood_risk",
] += 1

# ===== 3) 邊坡風險 =====
# 高海拔且坡度越大，邊坡或土石流相關風險越高
high_elev = g["mean_elevation"] > q75

g.loc[
    high_elev & (g["slope"] > 10) & (g["slope"] <= 20),
    "slope_risk",
] += 1

g.loc[
    high_elev & (g["slope"] > 20) & (g["slope"] <= 30),
    "slope_risk",
] += 2

g.loc[
    high_elev & (g["slope"] > 30),
    "slope_risk",
] += 3

# ===== 淹水總風險 =====
# 為避免後續使用外部淹水資料驗證時產生循環論證，
# total_flood_risk 僅使用地形與河川條件，不納入歷史淹水資料。
g["total_flood_risk"] = (
    g["terrain_flood_risk"]
    + g["river_flood_risk"]
)

display(
    g[
        [
            "grid_id",
            "terrain_flood_risk",
            "river_flood_risk",
            "total_flood_risk",
            "slope_risk",
        ]
    ]
    .sort_values("total_flood_risk", ascending=False)
    .head(20)
)

In [ ]:
# Cell 6: 輸出兩個檔案
# 1) GeoJSON（保留 geometry, EPSG:3826）
geojson_path = "output/Taipei_grid_full_risk.geojson"
g.to_file(geojson_path, driver="GeoJSON", encoding="utf-8")

# 2) CSV（改成經緯度欄位）
g_wgs84 = g.to_crs(epsg=4326).copy()

# 注意：因為 EPSG:4326 是經緯度座標，這裡只是用來輸出格網中心點 lon/lat
cent = g_wgs84.geometry.centroid
g_wgs84["lon"] = cent.x
g_wgs84["lat"] = cent.y

csv_cols = [
    "grid_id", "lon", "lat",
    "mean_elevation", "slope",
    "river_area_m2", "grid_area_m2", "river_ratio", "dist_to_river_m", "has_river",
    "terrain_flood_risk", "river_flood_risk",
    "total_flood_risk", "slope_risk"
]

g_wgs84[csv_cols].to_csv(
    "output/Taipei_grid_full_risk.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", geojson_path)
print("Saved: output/Taipei_grid_full_risk.csv")

### Taipei Pre Map 圖層
---
- `Total Flood Risk`
  - 顯示每個 grid 的總淹水風險分數
- `Slope Risk`
  - 顯示每個 grid 的邊坡風險分數
- `Road: expressway`
  - 快速道路圖層
- `Road: arterial`
  - 主要幹道圖層
- `Road: bridge`
  - 橋梁道路圖層
- `Road: residential`
  - 住宅道路圖層
- `Road: tunnel`
  - 隧道與地下道路圖層
- `Road: service`
  - 服務道路圖層
- `Taipei Boundary`
  - 臺北市行政邊界
- `Shelters`
  - 臺北市室內避難所
- `Grid Tooltip`
  - 滑鼠移到 grid 時顯示各項風險分數

In [ ]:
# Cell X: Taipei Interactive Risk Map
# total_flood_risk = terrain_flood_risk + river_flood_risk

import folium
import geopandas as gpd
import pandas as pd

# ===== 0) Prepare data =====
grid_wgs84 = g.to_crs(epsg=4326).copy()
edges_wgs84 = edges_gdf.to_crs(epsg=4326).copy()
shelters_wgs84 = matched_unique.to_crs(epsg=4326).copy()

# Taipei boundary
town_path = "data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp"
town_gdf = gpd.read_file(town_path)
town_gdf.columns = [str(c).strip().strip("'").strip('"') for c in town_gdf.columns]

if "COUNTYCODE" in town_gdf.columns:
    tp_town = town_gdf[
        town_gdf["COUNTYCODE"].astype(str).str.contains("63000", na=False)
    ].copy()
elif "COUNTYNAME" in town_gdf.columns:
    tp_town = town_gdf[
        town_gdf["COUNTYNAME"].astype(str).str.contains("臺北市|台北市", na=False, regex=True)
    ].copy()
else:
    tp_town = town_gdf.copy()

tp_town_wgs84 = tp_town.to_crs(epsg=4326)

# ===== 1) Base map =====
minx, miny, maxx, maxy = grid_wgs84.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="cartodbpositron",
)

# 讓 Grid Tooltip 永遠在最上層，且可以接收滑鼠事件
folium.map.CustomPane(
    "grid_tooltip_pane",
    z_index=900,
    pointer_events=True,
).add_to(m)

# ===== 2) Risk grid layers =====
folium.Choropleth(
    geo_data=grid_wgs84[["grid_id", "total_flood_risk", "geometry"]].to_json(),
    data=grid_wgs84[["grid_id", "total_flood_risk"]],
    columns=["grid_id", "total_flood_risk"],
    key_on="feature.properties.grid_id",
    fill_color="OrRd",
    fill_opacity=0.60,
    line_opacity=0.15,
    nan_fill_color="lightgray",
    legend_name="Total Flood Risk",
    name="Total Flood Risk",
    highlight=True,
).add_to(m)

folium.Choropleth(
    geo_data=grid_wgs84[["grid_id", "slope_risk", "geometry"]].to_json(),
    data=grid_wgs84[["grid_id", "slope_risk"]],
    columns=["grid_id", "slope_risk"],
    key_on="feature.properties.grid_id",
    fill_color="YlGnBu",
    fill_opacity=0.60,
    line_opacity=0.15,
    nan_fill_color="lightgray",
    legend_name="Slope Risk",
    name="Slope Risk",
    highlight=True,
    show=False,
).add_to(m)

# ===== 3) Road layers =====
color_map = {
    "expressway": "#e31a1c",
    "arterial": "#1f78b4",
    "bridge": "#ff7f00",
    "residential": "#33a02c",
    "tunnel": "#6a3d9a",
    "service": "#999999",
}

edges_plot = edges_wgs84.copy()

if "road_type" in edges_plot.columns:
    edges_plot["road_type_std"] = (
        edges_plot["road_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            "underground": "tunnel",
            "underground_road": "tunnel",
            "underpass": "tunnel",
        })
    )
else:
    edges_plot["road_type_std"] = "residential"

for rt, color in color_map.items():
    sub = edges_plot[edges_plot["road_type_std"] == rt].copy()
    if sub.empty:
        continue

    fg = folium.FeatureGroup(name=f"Road: {rt}", show=True)

    folium.GeoJson(
        sub[["geometry", "road_type_std"]].to_json(),
        style_function=lambda x, c=color: {
            "color": c,
            "weight": 1.4,
            "opacity": 0.9,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["road_type_std"],
            aliases=["Road Type"],
            labels=True,
            sticky=False,
        ),
    ).add_to(fg)

    fg.add_to(m)

# ===== 4) Taipei boundary =====
folium.GeoJson(
    tp_town_wgs84[["geometry"]].to_json(),
    name="Taipei Boundary",
    style_function=lambda x: {
        "color": "#000000",
        "weight": 2.2,
        "fillOpacity": 0,
    },
).add_to(m)

# ===== 5) Shelters =====
def safe_get(row, col, default="N/A"):
    if col in row.index and pd.notna(row[col]):
        return str(row[col])
    return default


shelter_fg = folium.FeatureGroup(name="Shelters", show=True)

for _, row in shelters_wgs84.iterrows():
    name = safe_get(row, "避難收容處所名稱")
    addr = safe_get(row, "避難收容處所地址")
    cap = safe_get(row, "預計收容人數")
    town = safe_get(row, "縣市及鄉鎮市區")
    hazard = safe_get(row, "適用災害類別")
    indoor = safe_get(row, "室內")
    outdoor = safe_get(row, "室外")
    sid = safe_get(row, "shelter_id")

    popup_html = f"""
    <b>避難收容處所名稱:</b> {name}<br>
    <b>Shelter ID:</b> {sid}<br>
    <b>行政區:</b> {town}<br>
    <b>地址:</b> {addr}<br>
    <b>預計收容人數:</b> {cap}<br>
    <b>適用災害類別:</b> {hazard}<br>
    <b>室內:</b> {indoor}<br>
    <b>室外:</b> {outdoor}
    """

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3.8,
        color="#111111",
        weight=1,
        fill=True,
        fill_color="#ffff33",
        fill_opacity=0.95,
        popup=folium.Popup(popup_html, max_width=380),
        tooltip=name,
    ).add_to(shelter_fg)

shelter_fg.add_to(m)

# ===== 6) Grid Tooltip: must be added last and on top pane =====
folium.GeoJson(
    grid_wgs84[
        [
            "grid_id",
            "terrain_flood_risk",
            "river_flood_risk",
            "total_flood_risk",
            "slope_risk",
            "geometry",
        ]
    ].to_json(),
    name="Grid Tooltip",
    pane="grid_tooltip_pane",
    style_function=lambda x: {
        "color": "#666666",
        "weight": 0.35,
        "opacity": 0.55,
        "fillColor": "#ffffff",
        "fillOpacity": 0.01,
    },
    highlight_function=lambda x: {
        "color": "#000000",
        "weight": 1.3,
        "opacity": 1,
        "fillColor": "#ffffff",
        "fillOpacity": 0.08,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "grid_id",
            "terrain_flood_risk",
            "river_flood_risk",
            "total_flood_risk",
            "slope_risk",
        ],
        aliases=[
            "Grid ID",
            "Terrain Flood Risk",
            "River Flood Risk",
            "Total Flood Risk",
            "Slope Risk",
        ],
        labels=True,
        sticky=False,
    ),
    show=True,
    control=True,
).add_to(m)

# ===== 7) Control & save =====
folium.LayerControl(collapsed=False).add_to(m)

map_path = "output/Taipei_pre_map.html"
m.save(map_path)

print("Saved:", map_path)

### Road-Grid Segment ID
---
此步驟將道路依照 `road_type` 進行分類與編號，並將道路切分到不同 grid 中，產生唯一的 `road_grid_id`。

首先，每一條道路會依照其道路類型建立 `road_base_id`。在同一種 `road_type` 內，所有道路會從 1 開始依序編號。

`road_base_id` 的命名格式為：

`road_type_道路序號`

例如：

| road_type | road_no_in_type | road_base_id |
|---|---:|---|
| bridge | 1 | `bridge_01` |
| arterial | 15 | `arterial_15` |
| residential | 230 | `residential_230` |

接著，將道路與臺北市 grid 進行空間疊圖。若同一條道路穿過多個 grid，該道路會被切分成多個 road-grid segment。每一個 road-grid segment 代表「某一條道路在某一個 grid 內的部分」。

最後，將 `road_base_id` 與該分段所在的 `grid_id` 結合，形成唯一的 `road_grid_id`。

`road_grid_id` 的命名格式為：

`road_base_id_ggrid_id`

例如：

| road_base_id | grid_id | road_grid_id |
|---|---:|---|
| `bridge_01` | 1 | `bridge_01_g01` |
| `bridge_01` | 2 | `bridge_01_g02` |
| `arterial_15` | 23 | `arterial_15_g23` |

其中：

| 欄位 | 說明 |
|---|---|
| `road_type` | 道路類型 |
| `road_no_in_type` | 同一道路類型內的道路序號 |
| `road_base_id` | 道路本身的 ID |
| `grid_id` | 道路分段所在的 grid 編號 |
| `road_grid_id` | 道路與 grid 交集後的唯一分段 ID |
| `segment_length_m` | 該道路分段在 grid 內的長度 |

因此，`road_grid_id` 可用來唯一識別「某一條道路在某一個 grid 內的分段」，並作為後續道路災害影響、通行速度與 travel time 計算的主要鍵值。

In [ ]:
# Cell Final: 將道路依 road_type 編號，並切到 grid 內產生 road_grid_id
# 例：第一條 bridge 穿過 grid 1、grid 2 -> bridge_01_g01, bridge_01_g02

import pandas as pd
import geopandas as gpd
import osmnx as ox

TARGET_CRS = "EPSG:3826"

# 如果 edges_gdf 不存在，就從 G 重新轉出來
if "edges_gdf" not in globals():
    edges_gdf = ox.graph_to_gdfs(G, nodes=False).copy()

# 1) 準備道路資料
roads = edges_gdf.copy().to_crs(TARGET_CRS)

# OSMnx edges 通常 index 是 u, v, key；轉成欄位方便保留
if not isinstance(roads.index, pd.RangeIndex):
    roads = roads.reset_index()

roads = roads[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

# 確保有 road_type
if "road_type" not in roads.columns:
    roads["road_type"] = "residential"

roads["road_type"] = (
    roads["road_type"]
    .fillna("residential")
    .astype(str)
    .str.strip()
    .str.lower()
)

# 給 road_type 做 ID 用名稱
roads["road_type_id"] = (
    roads["road_type"]
    .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
    .str.strip("_")
)

# 2) 每種 road_type 各自從 1 開始編號
sort_cols = [c for c in ["road_type_id", "u", "v", "key"] if c in roads.columns]
roads = roads.sort_values(sort_cols).reset_index(drop=True)

roads["road_no_in_type"] = roads.groupby("road_type_id").cumcount() + 1

roads["road_base_id"] = roads.apply(
    lambda r: f"{r['road_type_id']}_{int(r['road_no_in_type']):02d}",
    axis=1
)

# 3) 準備 grid
grid_for_road = g[["grid_id", "geometry"]].copy().to_crs(TARGET_CRS)

# 只保留要輸出的道路欄位，避免 GeoJSON 不喜歡 list/dict 欄位
road_cols = [
    c for c in [
        "u", "v", "key",
        "road_type", "road_type_id", "road_no_in_type", "road_base_id",
        "length", "speed_kph", "travel_time",
        "geometry"
    ]
    if c in roads.columns
]

roads_for_overlay = roads[road_cols].copy()

# 4) 將道路切到 grid 範圍內
road_grid_segments = gpd.overlay(
    roads_for_overlay,
    grid_for_road,
    how="intersection",
    keep_geom_type=True
)

# 5) 計算每段在 grid 內的長度
road_grid_segments["segment_length_m"] = road_grid_segments.geometry.length
road_grid_segments = road_grid_segments[
    road_grid_segments["segment_length_m"] > 0
].copy()

# 6) 產生 road_grid_id
road_grid_segments["road_grid_id"] = road_grid_segments.apply(
    lambda r: f"{r['road_base_id']}_g{int(r['grid_id']):02d}",
    axis=1
)

# 7) 調整欄位順序
first_cols = [
    "road_grid_id",
    "road_base_id",
    "road_type",
    "road_no_in_type",
    "grid_id",
    "segment_length_m"
]

other_cols = [
    c for c in road_grid_segments.columns
    if c not in first_cols + ["geometry"]
]

road_grid_segments = road_grid_segments[
    first_cols + other_cols + ["geometry"]
]

# 8) 輸出
out_path = "output/taipei_road_grid_segments.geojson"
road_grid_segments.to_file(out_path, driver="GeoJSON", encoding="utf-8")

print("Saved:", out_path)
print("road-grid segments:", len(road_grid_segments))

display(
    road_grid_segments
    .sort_values(["road_type", "road_no_in_type", "grid_id"])
    [[
        "road_grid_id",
        "road_base_id",
        "road_type",
        "road_no_in_type",
        "grid_id",
        "segment_length_m"
    ]]
    .head(30)
)